# micrograd exercises

1. watch the [micrograd video](https://www.youtube.com/watch?v=VMj-3S1tku0) on YouTube
2. come back and complete these exercises to level up :)

## section 1: derivatives

In [1]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos

def f(a, b, c):
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))

6.336362190988558


In [2]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29

def gradf(a, b, c):
  grad_a = -3*a**2 - 0.5*a**-0.5
  grad_b = 3 * cos(3*b) + 2.5*b**1.5
  grad_c = c**-2
  return [grad_a, grad_b, grad_c] # todo, return [df/da, df/db, df/dc]

# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390593273
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027111255
OK for dim 2: expected 0.0625, yours returns 0.0625


In [3]:
# now estimate the gradient numerically without any calculus, using
# the approximation we used in the video.
# you should not call the function df from the last cell

small_step = 0.000001
def f_a(a, b, c):
  return -(a+small_step)**3 + sin(3*b) - 1.0/c + b**2.5 - (a+small_step)**0.5

def f_b(a, b, c):
  return -a**3 + sin(3*(b+small_step)) - 1.0/c + (b+small_step)**2.5 - a**0.5

def f_c(a, b, c):
  return -a**3 + sin(3*b) - 1.0/(c+small_step) + b**2.5 - a**0.5


# -----------
numerical_grad = [(f_a(2, 3, 4)-f(2, 3, 4))/small_step, (f_b(2, 3, 4)-f(2, 3, 4))/small_step, (f_c(2, 3, 4)-f(2, 3, 4))/small_step] # TODO
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353559348809995
OK for dim 1: expected 10.25699027111255, yours returns 10.256991666679482
OK for dim 2: expected 0.0625, yours returns 0.062499984743169534


In [4]:
# there is an alternative formula that provides a much better numerical
# approximation to the derivative of a function.
# learn about it here: https://en.wikipedia.org/wiki/Symmetric_derivative
# implement it. confirm that for the same step size h this version gives a
# better approximation.
small_step = 0.000001
a = 2
b = 3
c = 4

# -----------
numerical_grad2 = [(f(a+small_step,b,c)-f(a-small_step,b,c))/(2*small_step), (f(a,b+small_step,c)-f(a,b-small_step,c))/(2*small_step), (f(a,b,c+small_step)-f(a,b,c-small_step))/(2*small_step)] # TODO
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553391353245
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027401572
OK for dim 2: expected 0.0625, yours returns 0.06250000028629188


## section 2: support for softmax

In [70]:
# Value class starter code, with many functions taken out
from math import exp, log

class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other): # exactly as in the video
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  def __radd__(self, other): # other + self
    return self + other

  def __mul__(self, other): # handles Value * (Value or float)
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out

  def __rmul__(self, other): # other * self
    return self * other

  def __neg__(self): # -self, correctly implemented using __mul__
    return self * -1

  # ------
  # re-implement all the other functions needed for the exercises below
  # your code here
  # TODO
  def exp(self): # exactly as in the video, with corrected backward pass
    out = Value(exp(self.data), (self, ), 'exp')

    def _backward():
      self.grad += out.data * out.grad # d(exp(x))/dx = exp(x)
    out._backward = _backward
    return out

  def __truediv__(self, other): # handles Value / (Value or float)
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data / other.data, (self, other), '/')

    def _backward():
      self.grad += (1.0 / other.data) * out.grad
      other.grad += (-self.data / (other.data**2)) * out.grad
    out._backward = _backward
    return out

  def log(self): # exactly as in the video, with corrected backward pass
    out = Value(log(self.data), (self, ), 'log')

    def _backward():
      self.grad += (1.0 / self.data) * out.grad # d(log(x))/dx = 1/x
    out._backward = _backward
    return out


  # ------

  def backward(self): # exactly as in video
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [71]:
# without referencing our code/video __too__ much, make this cell work
# you'll have to implement (in some cases re-implemented) a number of functions
# of the Value object, similar to what we've seen in the video.
# instead of the squared error loss this implements the negative log likelihood
# loss, which is very often used in classification.

# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
  counts = [logit.exp() for logit in logits]

  # The denominator needs to be a Value object for gradients to flow
  denominator = sum(counts) # This will now use Value.__add__ because of __radd__
  #print(denominator.data)
  out = [c / denominator for c in counts]
  return out

# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log() # dim 3 acts as the label for this input example
loss.backward()
print(loss)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")

Value(data=2.1755153626167147)
OK for dim 0: expected 0.041772570515350445, yours returns 0.041772570515350445
OK for dim 1: expected 0.8390245074625319, yours returns 0.8390245074625319
OK for dim 2: expected 0.005653302662216329, yours returns 0.005653302662216329
OK for dim 3: expected -0.8864503806400986, yours returns -0.886450380640099


In [72]:
# verify the gradient using the torch library
# torch should give you the exact same gradient
import torch

# Convert logits to torch tensors with requires_grad=True
t_logits = torch.tensor([l.data for l in logits], dtype=torch.float64, requires_grad=True)

# Perform softmax and negative log-likelihood loss using torch
t_probs = torch.softmax(t_logits, dim=0)
t_loss = -torch.log(t_probs[3])

# Backpropagate to get gradients
t_loss.backward()

# Print torch gradients and compare with our Value gradients
print("\n--- Torch Gradients ---")
for i, l in enumerate(logits):
    print(f"dim {i}: {t_logits.grad[i].item()}")

print("\n--- Comparison ---")
for dim in range(4):
    ok = 'OK' if abs(logits[dim].grad - t_logits.grad[dim].item()) < 1e-5 else 'WRONG!'
    print(f"{ok} for dim {dim}: Value: {logits[dim].grad}, Torch: {t_logits.grad[dim].item()}")


--- Torch Gradients ---
dim 0: 0.04177257051535046
dim 1: 0.8390245074625321
dim 2: 0.00565330266221633
dim 3: -0.8864503806400987

--- Comparison ---
OK for dim 0: Value: 0.041772570515350445, Torch: 0.04177257051535046
OK for dim 1: Value: 0.8390245074625319, Torch: 0.8390245074625321
OK for dim 2: Value: 0.005653302662216329, Torch: 0.00565330266221633
OK for dim 3: Value: -0.886450380640099, Torch: -0.8864503806400987
